# Phase 1: Resolution study

## Section 1 - Purpose and overview

This study decides the input resolution of C-RADIO: 1024x1024, the native size of the headset frames, or 512x512.

Frames have entered the model at 1024 so far, but this was never a motivated choice. Perception takes about 400 ms per frame at 1024, almost all of it in C-RADIO (pipeline, Section 12). In a Vision Transformer the cost of global attention grows with the square of the number of tokens (C-RADIOv4 tech report, Sect. 3.2). At 512 the grid has 32x32 patches instead of 64x64, four times fewer tokens, so a large gain in time is expected.

Both resolutions are legitimate for this model. It was trained by sampling the input resolution from a set that contains both 512 and 1024 (tech report, Sect. 2.2). Its configuration files give 512x512 as the preferred resolution (config.json, common.py), but the meaning of this value is not documented.

The question is what is lost. At 512 a patch covers 32x32 pixels of the frame, so a small piece covers about four times fewer patches. On a generic benchmark the loss is small: a linear probe of C-RADIOv4-H on ADE20k goes from 57.02 at 1024 to 55.20 at 512 (tech report, Table 2). In Cityscapes, halving the resolution of the labels hardly changes the IoU but clearly lowers the instance-level score, because small objects are hit hardest (Cordts et al. 2016, Sect. 3.2). Our pieces are small, so the general result cannot be assumed.

Everything except the resolution stays fixed: the 8 reference frames, the labelling rule, the prototypes, the classifier, the background margins (0 and 0.03), the validation frames and the bootstrap resamples. At 512 the reference patches are labelled with the same coverage rule, applied to the 32x32 pixels that each patch sees.

The two resolutions are evaluated on the same ground truth, the 64x64 patch grid at 1024. Each prediction at 512 is copied to the 2x2 block of 1024 patches it covers. Cityscapes evaluates low-resolution predictions in the same way, after resampling them to full resolution (Cordts et al. 2016, Sect. 3.2). The ground truth, the ignored patches and the exclusions are then identical, so a difference in IoU can only come from the predictions.

A coarser grid loses some IoU even when every piece is recognized, because its cells are larger at the borders. To separate this loss from a loss of recognition, the ground truth itself is reduced to the 32x32 grid by majority vote and copied back, as in Cityscapes (Sect. 3.2). Its score is the best that 512 can reach. We call it the oracle.

The decision follows a noninferiority design. The question is not whether 512 is better, but whether it is not worse by more than an acceptable amount, since it is faster (Piaggio et al. 2012, Rationale for Noninferiority or Equivalence Designs). The acceptable amount, the noninferiority margin, must be declared before running and justified by the use (Items 2b and 7a).

Phase 2 works at piece level: it checks a piece when it is picked and when it is mounted. A loss that only changes the borders of a piece does not matter there, while a missed piece does. The noninferiority margin is therefore the IoU lost when one typical piece is missed in one frame. It is computed from the ground truth alone in Section 4, before any feature is extracted at 512.

The rule has three conditions. First, the paired 95% bootstrap interval of the mIoU difference (512 minus 1024, arms merged) is compared with the noninferiority margin, at background margin 0 and 0.03 (Koehn 2004, Sect. 6; Piaggio et al. 2012, Item 12a and Fig. 1). If the whole interval lies above minus the margin, 512 is noninferior; if it lies below, 512 is inferior; if it contains it, the result is inconclusive. Second, no piece may have lost more than one missed piece: the interval of its IoU difference must not lie entirely below minus its own margin. Third, 512 must be measurably faster.

512 is chosen only if all three conditions hold. Otherwise, and also when the result is inconclusive, 1024 is kept: the change has to be shown, not assumed.

The arms are merged, as in the selection of the support set, because their identity will be decided at object level in Phase 2 (support-set study, Section 8). The 10-piece metric and the metric with the boundary ring ignored are still reported.

Every other choice was made at 1024: the support set, the provisional background margin and the labelling rule. The validation set is small, with 8 frames, 6 of them from the same recording. Choosing on validation makes the result optimistic, more so with many variants (Cawley and Talbot 2010, Sect. 4.4 and 5). For this reason only two resolutions are compared. 768, also a training resolution, is left out: its 48x48 grid does not align with the 64x64 ground truth, and adding it after seeing 512 would be a choice made on the results. The test sequence (assembly_1) is not read.

Section 3 checks that the shared code reproduces the previous results at 1024. Section 4 prepares the labels at both resolutions and computes the noninferiority margins and the oracle; it is committed before any feature is extracted at 512. Section 5 builds the prototypes at 512 and measures how fragile they are, Section 6 compares recognition, Section 7 latency, and Section 8 applies the rule.

------------------------------------------------------------------------------------------------------------------------------------------

References Section 1:

C-RADIOv4 tech report (Ranzinger et al. 2026), Sect. 2.2: the input resolution is sampled during training from {128, 192, 224, 256, 384, 432} and {512, 768, 1024, 1152}. Table 2: ADE20k linear probe of C-RADIOv4-H, 55.20 at 512 and 57.02 at 1024. Sect. 3.2: global attention has complexity O(T²) in the number of tokens T.

C-RADIOv4-SO400M model files (Hugging Face, commit c0457f5): config.json and common.py, preferred_resolution 512x512.

Cordts et al. 2016, Cityscapes, Sect. 3.2: labels subsampled by majority vote and resampled to full resolution give an upper bound for a fixed output resolution; subsampling by 2 hardly changes the IoU but lowers the instance-level score, which weighs small objects more. Sect. 3.4: downscaling consistently lowers performance, because of the large variation in object scale.

Piaggio et al. 2012, Reporting of Noninferiority and Equivalence Randomized Trials (CONSORT 2010 extension), JAMA 308(24):2594–2604. Rationale for Noninferiority or Equivalence Designs: noninferiority means not worse than the reference by more than an acceptable amount, and is of interest when the new option has another advantage. Item 2b: the noninferiority margin is stated with the rationale for its choice. Item 7a: the margin is preferably justified on practical (clinical) grounds; a margin too wide risks accepting a worse option, one too narrow gives inconclusive results. Item 12a and Fig. 1: the decision depends on where the whole confidence interval lies with respect to the margin.

Koehn 2004, Sect. 6: paired bootstrap resampling, two systems compared on the same resampled test sets.

Cawley and Talbot 2010, Sect. 4.4: selecting by optimizing a criterion on a finite sample over-fits, more with small samples and many choices; Sect. 5: a performance estimate on the data used for selection is optimistically biased.

## Section 2 - Configuration

All paths, frames and hyperparameters in one place. The values are the same as in the pipeline (Section 2.5) and in the support-set study (Section 2.4): this study changes only the input resolution.

In [ ]:
# 2.1 Dependencies: install once per environment with pip install -r requirements.txt (README, Setup)

In [ ]:
# 2.2 Imports
%load_ext autoreload
%autoreload 2

import sys
import time
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
import transformers
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
# 2.3 Paths and shared module
PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))   # makes src/perception importable from notebooks/

MODEL_DIR = PROJECT_ROOT / "models" / "c_radio_weights"
FRAMES_DIR = PROJECT_ROOT / "data" / "extracted_frames"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "phase_1_resolution_study"
PIPELINE_DIR = PROJECT_ROOT / "outputs" / "phase_1_perception"
PIPELINE_PROTOTYPES_PATH = PIPELINE_DIR / "prototypes.npz"                    # saved by the pipeline (v1.3): 8 reference frames, 1024
PIPELINE_SIMILARITIES_PATH = PIPELINE_DIR / "validation_similarities.npz"     # saved by the pipeline (v1.3): 8 validation frames, 1024

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {"MODEL_DIR": MODEL_DIR, "FRAMES_DIR": FRAMES_DIR, "SRC_DIR": SRC_DIR,
                   "PIPELINE_PROTOTYPES_PATH": PIPELINE_PROTOTYPES_PATH,
                   "PIPELINE_SIMILARITIES_PATH": PIPELINE_SIMILARITIES_PATH}.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"[{status}] {name}: {path}")

# Shared code (src/perception), the same used by the pipeline and the support-set study
from perception.classes import (CLASS_NAMES, BACKGROUND_ID, HAND_LABEL, PIECE_LABELS, IGNORE_ID,
                                CLASS_COLORS, PATCH_LABEL_NAMES)
from perception.features import FeatureExtractor
from perception.labels import sequence_of, check_annotation, rasterize_annotation, patch_label_grid
from perception.prototypes import build_prototypes, stack_prototypes, load_prototypes, classify_patches
from perception.metrics import (confusion_matrix, iou_from_confusions, mean_iou, ignore_boundary_ring,
                                MERGED_NAMES, MERGED_PIECES, upsample_labels, merge_arms,
                                frame_iou, frame_miou, bootstrap_miou, bootstrap_class_iou,
                                percentile_interval)
from perception.visualization import label_grid_to_rgb

In [ ]:
# 2.4 Hyperparameters
# Same values as the pipeline (Section 2.5) and the support-set study (Section 2.4).
# The only new parameters are the two resolutions and the latency runs.

# --- Feature extraction ---
DEVICE = "cuda"
DTYPE = torch.bfloat16
NATIVE_RESOLUTION = 1024        # headset frames; also the resolution of the ground truth (64x64 patch grid)
LOW_RESOLUTION = 512            # the alternative: a training resolution (tech report, Sect. 2.2), preferred in config.json
RESOLUTIONS = [NATIVE_RESOLUTION, LOW_RESOLUTION]
UPSAMPLE_FACTOR = NATIVE_RESOLUTION // LOW_RESOLUTION   # one 512 patch = a 2x2 block of 1024 patches
assert NATIVE_RESOLUTION % LOW_RESOLUTION == 0, "The two patch grids must align exactly"

# --- Patch labelling ---
MIN_LABEL_COVERAGE = 0.5        # same rule as the pipeline (Section 5), at both resolutions

# --- Classification ---
CONFIDENCE_MARGIN = 0.03        # background margin, provisional (pipeline, Section 9): not tuned here
EVAL_MARGINS = [0.0, CONFIDENCE_MARGIN]   # 0.0 = pure nearest prototype, independent of the provisional value

# --- Evaluation ---
N_BOOTSTRAP = 1000              # resamples of the validation frames (support-set study, Section 5.4)
BOUNDARY_RING = 1               # secondary metric: background patches within 1 patch of an object are ignored

# --- Latency (Section 7), same protocol as the pipeline, Section 12 ---
N_WARMUP, N_RUNS = 5, 50

# --- Reproducibility ---
RANDOM_SEED = 0
np.random.seed(RANDOM_SEED)
_ = torch.manual_seed(RANDOM_SEED)